Carolina Quantum Hacks 2026
Beginner Starter Notebook: Quantum Dreamscapes
This notebook is a starting environment for the Quantum Dreamscapes beginner challenge.

It is meant to help you:

build and simulate small quantum circuits
collect measurement results
turn those results into visual elements
experiment with your own artistic mapping
This notebook is intentionally open-ended. It gives you tools, not a finished project.

Suggested workflow
1. Build one or more small circuits.
2. Sample them many times.
3. Convert measurement counts into numbers, positions, shapes, colors, or other visual features.
4. Iterate until the visual style feels interesting and coherent.
5. Explain how the circuit influenced the artwork.

In [ ]:

# If you are running in Colab, uncomment the next line.
# !pip -q install qiskit qiskit-aer numpy matplotlib networkx scipy

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
rng = np.random.default_rng(7)

1. A few example circuit builders

In [ ]:
def circuit_superposition(n_qubits=4):
    qc = QuantumCircuit(n_qubits, n_qubits)
    for q in range(n_qubits):
        qc.h(q)
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def circuit_layered_phase(n_qubits=4):
    qc = QuantumCircuit(n_qubits, n_qubits)
    for q in range(n_qubits):
        qc.h(q)
        qc.rz((q + 1) * np.pi / 5, q)
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def circuit_ring_entangler(n_qubits=4):
    qc = QuantumCircuit(n_qubits, n_qubits)
    for q in range(n_qubits):
        qc.ry(np.pi / (q + 2), q)
    for q in range(n_qubits - 1):
        qc.cz(q, q + 1)
    qc.cz(n_qubits - 1, 0)
    for q in range(n_qubits):
        qc.h(q)
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

2. Run a circuit and look at the counts

In [ ]:
def sample_counts(qc, shots=2048):
    tqc = transpile(qc, sim)
    result = sim.run(tqc, shots=shots).result()
    return result.get_counts()

qc = circuit_layered_phase(5)
counts = sample_counts(qc, shots=3000)
counts

In [ ]:
def plot_histogram_simple(counts, top_k=16, title="Measurement counts"):
    items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
    labels = [k for k, _ in items]
    values = [v for _, v in items]
    plt.figure(figsize=(10, 4))
    plt.bar(labels, values)
    plt.xticks(rotation=45)
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_histogram_simple(counts, title="Example histogram")

3. Turn bitstrings into features

In [ ]:
def bitstring_to_int(s):
    return int(s, 2)

def counts_to_feature_table(counts):
    total = sum(counts.values())
    rows = []
    for bitstring, count in counts.items():
        x = bitstring_to_int(bitstring)
        prob = count / total
        ones = bitstring.count("1")
        transitions = sum(bitstring[i] != bitstring[i+1] for i in range(len(bitstring)-1))
        rows.append({
            "bitstring": bitstring,
            "count": count,
            "prob": prob,
            "value": x,
            "ones": ones,
            "transitions": transitions,
        })
    rows = sorted(rows, key=lambda r: r["count"], reverse=True)
    return rows

features = counts_to_feature_table(counts)
features[:5]

4. Example visual mapping: dream-like scatter field

In [ ]:
def dreamscape_scatter(counts, seed=0):
    rng = np.random.default_rng(seed)
    rows = counts_to_feature_table(counts)

    plt.figure(figsize=(8, 8))
    for row in rows:
        x = np.cos(row["value"]) * (1 + 3 * row["prob"]) + rng.normal(scale=0.08)
        y = np.sin(row["value"]) * (1 + 3 * row["prob"]) + rng.normal(scale=0.08)
        size = 1000 * row["prob"] + 50 * row["ones"]
        alpha = 0.2 + 0.7 * row["prob"]
        plt.scatter(x, y, s=size, alpha=alpha)

    plt.title("Example Dreamscape Scatter")
    plt.axis("off")
    plt.show()

dreamscape_scatter(counts, seed=2)

5. Example visual mapping: layered radial pattern

In [ ]:
def dreamscape_radial(counts):
    rows = counts_to_feature_table(counts)
    plt.figure(figsize=(8, 8))
    ax = plt.gca()
    ax.set_aspect("equal")

    for row in rows[:20]:
        theta = 2 * np.pi * row["value"] / max(1, 2**len(row["bitstring"]) - 1)
        radius = 1 + 3 * row["prob"] + 0.2 * row["transitions"]
        x = radius * np.cos(theta)
        y = radius * np.sin(theta)
        plt.plot([0, x], [0, y], alpha=0.35)
        plt.scatter([x], [y], s=500 * row["prob"] + 30, alpha=0.7)

    plt.title("Example Dreamscape Radial Mapping")
    plt.axis("off")
    plt.show()

dreamscape_radial(counts)

6. Team work area

In [ ]:
# TEAM WORK AREA

def your_circuit_here():
    qc = QuantumCircuit(5, 5)
    # add gates here
    qc.measure(range(5), range(5))
    return qc

# your_counts = sample_counts(your_circuit_here(), shots=4000)
# dreamscape_scatter(your_counts)